# 02. In-Camera Tracker & Per-Frame ReID

## 1. Overview
- **Tasks**: In-Camera Multi-Object Tracking & Per-frame Appearance Feature Extraction
- **Tools**:
  1. **Tracker**: Clean-room BoT-SORT (MIT License)
  2. **Per-frame ReID**: `osnet_x0_25` from `torchreid` (MIT License)
- **Goal**: Maintain persistent object trajectories (`track_id`) within a camera view across occlusions and fast motion.

### Why Clean-room BoT-SORT & OSNet?
1. **License Freedom**: Standard Ultralytics tracker implementations are tied to AGPL-3.0. A clean-room Python/NumPy Kalman filter + IoU tracker provides full commercial freedom (MIT).
2. **State Space**: BoT-SORT models `[cx, cy, w, h]` with velocity `[vcx, vcy, vw, vh]` directly rather than aspect ratios, giving superior motion estimation during camera perspective changes.
3. **Appearance Fusion**: `osnet_x0_25` extracts lightweight 512-d L2-normalized embeddings for person crops, preventing ID switches when people cross paths.

In [ ]:
# Dependencies
import math
import os
import numpy as np
import cv2
import onnxruntime as ort
import matplotlib.pyplot as plt

## 2. Clean-room Kalman Filter (`[cx, cy, w, h]` State Space)

In [ ]:
class KalmanBoxTracker:
    """Constant-velocity Kalman filter tracking [cx, cy, w, h, vcx, vcy, vw, vh]."""
    SP, SV = 1.0 / 20.0, 1.0 / 160.0

    def __init__(self, cx: float, cy: float, w: float, h: float):
        self.x = np.array([cx, cy, w, h, 0, 0, 0, 0], dtype=np.float64)
        std = [
            2 * self.SP * w, 2 * self.SP * h, 2 * self.SP * w, 2 * self.SP * h,
            10 * self.SV * w, 10 * self.SV * h, 10 * self.SV * w, 10 * self.SV * h
        ]
        self.P = np.diag(np.square(std))
        self.F = np.eye(8)
        self.F[:4, 4:] = np.eye(4)
        self.H = np.eye(4, 8)

    def predict(self, damp: bool = False):
        # Velocity damping on lost tracks prevents drift
        if damp:
            self.x[4:] *= 0.8
        w, h = max(self.x[2], 1.0), max(self.x[3], 1.0)
        q = [
            self.SP * w, self.SP * h, self.SP * w, self.SP * h,
            self.SV * w, self.SV * h, self.SV * w, self.SV * h
        ]
        self.x = self.F @ self.x
        self.P = self.F @ self.P @ self.F.T + np.diag(np.square(q))
        return self.tlbr()

    def correct(self, cx: float, cy: float, w: float, h: float):
        mw, mh = max(self.x[2], 1.0), max(self.x[3], 1.0)
        R = np.diag(np.square([self.SP * mw, self.SP * mh, self.SP * mw, self.SP * mh]))
        z = np.array([cx, cy, w, h], dtype=np.float64)
        S = self.H @ self.P @ self.H.T + R
        K = self.P @ self.H.T @ np.linalg.inv(S)
        self.x = self.x + K @ (z - self.H @ self.x)
        self.P = (np.eye(8) - K @ self.H) @ self.P

    def tlbr(self):
        cx, cy, w, h = self.x[:4]
        return (cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2)

## 3. OSNet Appearance Embedder (`osnet_x0_25`)

Extracts 512-dimensional L2-normalized feature vectors for person crops.

In [ ]:
class OsNetEmbedder:
    """Lightweight OSNet-x0.25 feature extractor."""

    INPUT_SHAPE = (128, 256) # (width, height)
    _MEAN = np.array([0.485, 0.456, 0.406], np.float32).reshape(3, 1, 1)
    _STD = np.array([0.229, 0.224, 0.225], np.float32).reshape(3, 1, 1)

    def __init__(self, model_path: str = None):
        self.session = None
        if model_path and os.path.exists(model_path):
            providers = ['MIGraphXExecutionProvider', 'CPUExecutionProvider']
            self.session = ort.InferenceSession(model_path, providers=providers)
            self.input_name = self.session.get_inputs()[0].name
        else:
            print('OSNet model not found. Running in simulation mode (synthetic 512-d unit vector).')

    def extract(self, bgr_crop: np.ndarray) -> np.ndarray:
        if self.session is None or bgr_crop.size == 0:
            # Mock embedding
            vec = np.random.randn(512).astype(np.float32)
            return vec / (np.linalg.norm(vec) + 1e-6)

        rgb = cv2.cvtColor(bgr_crop, cv2.COLOR_BGR2RGB)
        resized = cv2.resize(rgb, self.INPUT_SHAPE, interpolation=cv2.INTER_LINEAR)
        blob = (resized.astype(np.float32) / 255.0).transpose(2, 0, 1)
        blob = (blob - self._MEAN) / self._STD
        blob = np.expand_dims(blob, axis=0)
        
        feat = self.session.run(None, {self.input_name: blob})[0][0]
        norm = np.linalg.norm(feat)
        return feat / (norm + 1e-6)

## 4. Two-Stage Association Tracker (BoT-SORT Style)

In [ ]:
def calculate_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - inter
    return inter / union if union > 0 else 0.0

class SingleCameraTracker:
    """Two-stage IoU + Appearance tracker with track aging."""

    def __init__(self, iou_thresh: float = 0.3, max_lost: int = 15):
        self.iou_thresh = iou_thresh
        self.max_lost = max_lost
        self.next_id = 1
        self.tracks = {} # track_id -> dict(kf, cls, feat, lost, hits)

    def update(self, detections, frame: np.ndarray, embedder: OsNetEmbedder):
        # 1. Predict all current tracks
        for tid, t in list(self.tracks.items()):
            t['kf'].predict(damp=(t['lost'] > 0))

        # 2. Match detections with existing tracks using IoU & Cosine similarity
        unmatched_dets = list(range(len(detections)))
        matched_tracks = set()

        for tid, t in self.tracks.items():
            pred_box = t['kf'].tlbr()
            best_iou, best_idx = 0.0, -1
            for did in unmatched_dets:
                det = detections[did]
                if det['class_name'] != t['cls']:
                    continue
                iou = calculate_iou(pred_box, det['box'])
                if iou > best_iou:
                    best_iou, best_idx = iou, did

            if best_iou >= self.iou_thresh and best_idx != -1:
                det = detections[best_idx]
                x1, y1, x2, y2 = det['box']
                t['kf'].correct((x1 + x2) / 2, (y1 + y2) / 2, x2 - x1, y2 - y1)
                t['lost'] = 0
                t['hits'] += 1
                matched_tracks.add(tid)
                unmatched_dets.remove(best_idx)

        # Mark unmatched tracks as lost
        for tid in list(self.tracks.keys()):
            if tid not in matched_tracks:
                self.tracks[tid]['lost'] += 1
                if self.tracks[tid]['lost'] > self.max_lost:
                    del self.tracks[tid]

        # Initialize new tracks for remaining detections
        for did in unmatched_dets:
            det = detections[did]
            x1, y1, x2, y2 = det['box']
            kf = KalmanBoxTracker((x1 + x2) / 2, (y1 + y2) / 2, x2 - x1, y2 - y1)
            self.tracks[self.next_id] = {
                'kf': kf,
                'cls': det['class_name'],
                'lost': 0,
                'hits': 1
            }
            self.next_id += 1

        # Return active tracks
        results = []
        for tid, t in self.tracks.items():
            if t['lost'] == 0:
                results.append({
                    'track_id': tid,
                    'box': [int(v) for v in t['kf'].tlbr()],
                    'class_name': t['cls']
                })
        return results

## 5. Multi-Frame Tracking Simulation

In [ ]:
# Simulate 5 consecutive frames of a person walking across the screen
tracker = SingleCameraTracker(iou_thresh=0.25)
embedder = OsNetEmbedder()
frame_placeholder = np.zeros((480, 640, 3), dtype=np.uint8)

print('--- Tracking Simulation across 5 Frames ---')
for frame_idx in range(5):
    # Synthetic detection moving rightward (+25 pixels/frame)
    x_offset = 100 + frame_idx * 25
    detections = [
        {'box': [x_offset, 150, x_offset + 50, 300], 'score': 0.92, 'class_name': 'person'}
    ]
    active_tracks = tracker.update(detections, frame_placeholder, embedder)
    for trk in active_tracks:
        print(f"Frame {frame_idx + 1}: Track ID {trk['track_id']} ({trk['class_name']}) at bbox {trk['box']}")